## Imports

In [1]:
import math
import itertools
from typing import Dict, Tuple, List, Set, Optional

import numpy as np

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector

try:
    from qiskit_aer import Aer
    HAS_AER = True
except Exception:
    HAS_AER = False

## Global Configuration

These variables replace the old `__init__` parameters and are used by all functions below.

In [2]:
# ── Problem parameters (set these before calling anything) ──
customers: Dict[int, Tuple[float, float]] = {}
Nv: int = 0
C: int = 0
depot: Tuple[float, float] = (0.0, 0.0)
q: int = 6            # neighborhood_qubits
shots: int = 2048
gamma: float = 0.8
layers: int = 1
seed: int = 0

# ── Derived state (populated by init_problem) ──
coords: Dict[int, Tuple[float, float]] = {}
customer_ids: List[int] = []
nodes: List[int] = []
n: int = 0
dist: Dict[Tuple[int, int], float] = {}

## Initialization

In [3]:
def init_problem(
    _customers: Dict[int, Tuple[float, float]],
    _Nv: int,
    _C: int,
    _depot: Tuple[float, float] = (0.0, 0.0),
    neighborhood_qubits: int = 6,
    _shots: int = 2048,
    _gamma: float = 0.8,
    _layers: int = 1,
    _seed: int = 0,
):
    """Initialize all global problem state (replaces the old __init__)."""
    global customers, Nv, C, depot, q, shots, gamma, layers, seed
    global coords, customer_ids, nodes, n, dist

    customers = _customers
    Nv = _Nv
    C = _C
    depot = _depot
    q = neighborhood_qubits
    shots = _shots
    gamma = _gamma
    layers = _layers
    seed = _seed

    coords = {0: depot, **customers}
    customer_ids = sorted(customers.keys())
    nodes = [0] + customer_ids
    n = len(customer_ids)

    dist = {
        (i, j): _euclidean(i, j)
        for i in nodes for j in nodes
    }

## Geometry

In [4]:
def _euclidean(i: int, j: int) -> float:
    x1, y1 = coords[i]
    x2, y2 = coords[j]
    return math.hypot(x1 - x2, y1 - y2)

In [5]:
def route_cost(route: List[int]) -> float:
    if not route:
        return 0.0
    total = dist[(0, route[0])]
    for a, b in zip(route[:-1], route[1:]):
        total += dist[(a, b)]
    total += dist[(route[-1], 0)]
    return total

## Exact Ordering — Held-Karp TSP

In [6]:
def best_order_for_subset(subset: List[int]) -> Tuple[List[int], float]:
    subset = list(subset)
    m = len(subset)
    if m == 0:
        return [], 0.0
    if m == 1:
        return subset[:], route_cost(subset)

    idx_to_customer = {i: subset[i] for i in range(m)}
    DP = {}
    parent = {}

    for j in range(m):
        mask = 1 << j
        cj = idx_to_customer[j]
        DP[(mask, j)] = dist[(0, cj)]
        parent[(mask, j)] = None

    for mask in range(1, 1 << m):
        for j in range(m):
            if not (mask & (1 << j)):
                continue
            if (mask, j) not in DP:
                continue
            for nxt in range(m):
                if mask & (1 << nxt):
                    continue
                new_mask = mask | (1 << nxt)
                cj = idx_to_customer[j]
                cn = idx_to_customer[nxt]
                cand = DP[(mask, j)] + dist[(cj, cn)]
                if (new_mask, nxt) not in DP or cand < DP[(new_mask, nxt)]:
                    DP[(new_mask, nxt)] = cand
                    parent[(new_mask, nxt)] = j

    full = (1 << m) - 1
    best_cost = float("inf")
    best_last = None

    for j in range(m):
        cj = idx_to_customer[j]
        cand = DP[(full, j)] + dist[(cj, 0)]
        if cand < best_cost:
            best_cost = cand
            best_last = j

    order_idx = []
    mask = full
    j = best_last
    while j is not None:
        order_idx.append(j)
        prev = parent[(mask, j)]
        mask ^= (1 << j)
        j = prev
    order_idx.reverse()

    order = [idx_to_customer[i] for i in order_idx]
    return order, best_cost

## Local Neighborhood

In [7]:
def nearest_neighbors(seed_node: int, pool: Set[int], k: int) -> List[int]:
    arr = list(pool)
    arr.sort(key=lambda c: dist[(seed_node, c)])
    return arr[:k]

## Scoring (classical decoding)

In [8]:
def subset_score(subset: List[int]) -> float:
    if len(subset) == 0:
        return -1e9
    if len(subset) > C:
        return -1e9

    _, route_c = best_order_for_subset(subset)

    pair_dispersion = sum(
        dist[(i, j)] for i, j in itertools.combinations(subset, 2)
    )
    depot_pull = sum(dist[(0, i)] for i in subset)

    return (
        8.0 * len(subset)
        - 1.0 * route_c
        - 0.07 * pair_dispersion
        - 0.03 * depot_pull
    )

## QUBO Encoding

In [9]:
def _build_qubo_coefficients(
    neighborhood: List[int],
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Encode subset scoring as a QUBO: f(x) = sum_i h_i x_i + sum_{i<j} J_ij x_i x_j

    Linear terms h_i:
        8.0 reward for including customer i
        minus depot round-trip cost (approximates individual routing cost)
        plus soft capacity penalty linear part: lambda*(1 - 2*C)

    Quadratic terms J_ij:
        Clarke-Wright savings: dist(0,i) + dist(0,j) - dist(i,j)
        minus pair dispersion penalty
        minus soft capacity penalty quadratic part: 2*lambda
    """
    q_len = len(neighborhood)
    h = np.zeros(q_len)
    J = np.zeros((q_len, q_len))
    lam = 5.0

    for i, ci in enumerate(neighborhood):
        h[i] = 8.0 - dist[(0, ci)]
        h[i] += lam * (1.0 - 2.0 * C)

    for i in range(q_len):
        for j in range(i + 1, q_len):
            ci, cj = neighborhood[i], neighborhood[j]
            savings = (
                dist[(0, ci)] + dist[(0, cj)] - dist[(ci, cj)]
            )
            J[i, j] = savings - 0.07 * dist[(ci, cj)] - 2.0 * lam

    return h, J

In [10]:
def _qubo_to_ising(
    h: np.ndarray,
    J: np.ndarray,
) -> Tuple[float, np.ndarray, np.ndarray]:
    """
    Convert QUBO (x_i in {0,1}) to Ising (z_i in {-1,+1}).
    Substitution: x_i = (1 - z_i) / 2
    """
    q_len = len(h)
    offset = 0.0
    h_z = np.zeros(q_len)
    J_z = np.zeros((q_len, q_len))

    for i in range(q_len):
        offset += h[i] / 2.0
        h_z[i] -= h[i] / 2.0

    for i in range(q_len):
        for j in range(i + 1, q_len):
            if abs(J[i, j]) < 1e-15:
                continue
            offset += J[i, j] / 4.0
            h_z[i] -= J[i, j] / 4.0
            h_z[j] -= J[i, j] / 4.0
            J_z[i, j] += J[i, j] / 4.0

    return offset, h_z, J_z

## DQI Circuit — Poly-depth, No Classical 2^q Enumeration

In [11]:
def _build_dqi_circuit(neighborhood: List[int]) -> QuantumCircuit:
    """
    Build the DQI circuit for the given neighborhood.

    Structure (one layer):
        |0>^q -> H^q -> PhaseOracle(gamma) -> H^q -> Measure

    Phase oracle implements e^{i*gamma*(sum h_z_i Z_i + sum J_z_ij Z_i Z_j)}:
        Single-qubit terms: Rz(-2*phi)
        Two-qubit terms:    CNOT(i,j) . Rz(-2*phi, j) . CNOT(i,j)

    Total circuit depth: O(q^2) for dense QUBO.
    """
    q_len = len(neighborhood)
    h_q, J_q = _build_qubo_coefficients(neighborhood)
    _, h_z, J_z = _qubo_to_ising(h_q, J_q)

    # Normalize Ising coefficients to [-1, 1]
    scale = max(np.max(np.abs(h_z)), np.max(np.abs(J_z)), 1e-12)
    h_z /= scale
    J_z /= scale

    qc = QuantumCircuit(q_len, q_len)
    qc.h(range(q_len))  # uniform superposition

    for _ in range(layers):
        # Phase oracle
        for i in range(q_len):
            if abs(h_z[i]) > 1e-12:
                qc.rz(-2.0 * gamma * h_z[i], i)

        for i in range(q_len):
            for j in range(i + 1, q_len):
                if abs(J_z[i, j]) < 1e-12:
                    continue
                qc.cx(i, j)
                qc.rz(-2.0 * gamma * J_z[i, j], j)
                qc.cx(i, j)

        # Interference / mixing
        qc.h(range(q_len))

    qc.measure(range(q_len), range(q_len))
    return qc

## Circuit Execution

In [12]:
def _run_circuit(qc: QuantumCircuit) -> dict:
    if HAS_AER:
        backend = Aer.get_backend("aer_simulator")
        tqc = transpile(qc, backend, seed_transpiler=seed)
        result = backend.run(
            tqc, shots=shots, seed_simulator=seed
        ).result()
        return result.get_counts()

    # Fallback: exact statevector simulation
    qc_copy = qc.copy()
    qc_copy.remove_final_measurements(inplace=True)
    sv = Statevector.from_instruction(qc_copy)
    probs = sv.probabilities_dict()
    return {
        k: int(round(v * shots))
        for k, v in probs.items()
        if v > 1e-12
    }

## Quantum Sampling + Classical Decoding

In [13]:
def _decode(
    candidate: List[int],
    must_include: Optional[int],
) -> List[int]:
    """
    Classical decoding: repair constraint violations in a measured bitstring.
    1. Enforce must_include
    2. Enforce capacity (|subset| <= C)
    """
    candidate = list(candidate)

    if must_include is not None and must_include not in candidate:
        candidate = [must_include] + candidate

    if len(candidate) > C:
        if must_include is not None:
            optional = [c for c in candidate if c != must_include]
            optional.sort(key=lambda c: dist[(must_include, c)])
            candidate = [must_include] + optional[: C - 1]
        else:
            candidate = candidate[: C]

    return candidate

In [14]:
def quantum_sample_subset(
    neighborhood: List[int],
    must_include: Optional[int] = None,
) -> List[int]:
    """
    Run the DQI circuit and apply classical decoding.
    Evaluate every distinct measured bitstring using the full
    subset_score function and return the best feasible subset.
    """
    qc = _build_dqi_circuit(neighborhood)
    counts = _run_circuit(qc)

    if not counts:
        return [must_include] if must_include is not None else []

    best_subset: Optional[List[int]] = None
    best_score = -float("inf")

    for bitstring in counts:
        bits = bitstring[::-1]
        raw = [neighborhood[idx] for idx, bit in enumerate(bits) if bit == "1"]

        decoded = _decode(raw, must_include)

        score = subset_score(decoded)
        if score > best_score:
            best_score = score
            best_subset = decoded

    return best_subset if best_subset else (
        [must_include] if must_include is not None else []
    )

## Build Full CVRP Solution

In [15]:
def build_solution() -> List[List[int]]:
    remaining = set(customer_ids)
    routes = []

    while remaining:
        seed_node = max(remaining, key=lambda c: dist[(0, c)])

        neighborhood = [seed_node] + nearest_neighbors(
            seed_node, remaining - {seed_node}, q - 1
        )
        neighborhood = list(dict.fromkeys(neighborhood))

        subset = quantum_sample_subset(
            neighborhood=neighborhood,
            must_include=seed_node,
        )
        subset = [c for c in subset if c in remaining]

        if len(subset) < min(C, len(remaining)):
            missing = min(C, len(remaining)) - len(subset)
            candidates = [
                c for c in neighborhood if c in remaining and c not in subset
            ]
            candidates.sort(key=lambda c: dist[(seed_node, c)])
            subset += candidates[:missing]

        order, _ = best_order_for_subset(subset)
        routes.append(order)
        for c in order:
            remaining.remove(c)

    return routes

## Classical Cleanup

In [16]:
def two_opt(route: List[int]) -> List[int]:
    best = route[:]
    improved = True
    while improved:
        improved = False
        base = route_cost(best)
        n_r = len(best)
        for i in range(n_r - 1):
            for j in range(i + 1, n_r):
                cand = best[:i] + best[i:j+1][::-1] + best[j+1:]
                c = route_cost(cand)
                if c + 1e-9 < base:
                    best = cand
                    base = c
                    improved = True
    return best

In [17]:
def improve_routes(routes: List[List[int]], passes: int = 10) -> List[List[int]]:
    routes = [two_opt(r) for r in routes]

    for _ in range(passes):
        changed = False
        for a in range(len(routes)):
            for b in range(a + 1, len(routes)):
                ra, rb = routes[a], routes[b]
                base = route_cost(ra) + route_cost(rb)
                best_pair = None
                best_delta = 0.0

                for ia, ca in enumerate(ra):
                    for ib, cb in enumerate(rb):
                        na = ra[:]
                        nb = rb[:]
                        na[ia], nb[ib] = cb, ca
                        na, cna = best_order_for_subset(na)
                        nb, cnb = best_order_for_subset(nb)
                        new_cost = cna + cnb
                        delta = base - new_cost
                        if delta > best_delta + 1e-9:
                            best_delta = delta
                            best_pair = (na, nb)

                if best_pair is not None:
                    routes[a], routes[b] = best_pair
                    changed = True

        routes = [two_opt(r) for r in routes]
        if not changed:
            break

    return routes

## Solve

In [18]:
def solve(do_classical_cleanup: bool = True, verbose: bool = True):
    routes = build_solution()
    init_cost = sum(route_cost(r) for r in routes)

    if do_classical_cleanup:
        routes = improve_routes(routes)

    final_cost = sum(route_cost(r) for r in routes)

    if verbose:
        print("DQI (Decoded Quantum Interferometry) CVRP solution")
        print(f"Neighborhood qubits per DQI call: {q}")
        print(f"DQI layers: {layers}  gamma: {gamma}")
        print(f"Shots per quantum call: {shots}")
        print()
        for k, r in enumerate(routes, start=1):
            print(
                f"Vehicle {k}: [0, " + ", ".join(map(str, r)) + ", 0] "
                f"load={len(r)} cost={route_cost(r):.4f}"
            )
        print(f"\nInitial total cost: {init_cost:.4f}")
        print(f"Final total cost:   {final_cost:.4f}")

    return routes, final_cost

## Example Instance

## Benchmarking and Experimental Sections

The remaining sections keep the original benchmark code exactly as implemented. They evaluate DQI across multiple CVRP instances, seeds, qubit counts, and shot settings, and export CSV summaries and plots for later comparison.

These cells are preserved without changing their logic so the notebook remains faithful to the original implementation.

In [19]:
from pathlib import Path
import csv
import matplotlib.pyplot as plt

# =========================================================
# Benchmark DQI on 6 CVRP instances
# - trials are different qubit counts: q = 1, 2, ..., 7
# - capacity stays fixed for each instance
# - fixed seed for all trials
# - outputs:
#     1) dqi_benchmark_summary.csv
#     2) dqi_benchmark_detailed.csv
#     3) one line chart per instance: cost vs qubits
# =========================================================

OUTPUT_DIR = Path(".")
PLOTS_DIR = OUTPUT_DIR / "dqi_qubit_plots"
PLOTS_DIR.mkdir(exist_ok=True)

SUMMARY_CSV = OUTPUT_DIR / "dqi_benchmark_summary.csv"
DETAILED_CSV = OUTPUT_DIR / "dqi_benchmark_detailed.csv"

FIXED_SEED = 101
SHOTS = 2048
GAMMA = 0.8
LAYERS = 1
MAX_QUBITS_TO_TEST = 7
DO_CLASSICAL_CLEANUP = True
VERBOSE_SOLVE = False

INSTANCES = [
    {
        "instance_number": 1,
        "Nv": 2,
        "C": 5,
        "depot": (0, 0),
        "customers": {
            1: (-2, 2),
            2: (-5, 8),
            3: (2, 3),
        },
    },
    {
        "instance_number": 2,
        "Nv": 2,
        "C": 2,
        "depot": (0, 0),
        "customers": {
            1: (-2, 2),
            2: (-5, 8),
            3: (2, 3),
        },
    },
    {
        "instance_number": 3,
        "Nv": 3,
        "C": 2,
        "depot": (0, 0),
        "customers": {
            1: (-2, 2),
            2: (-5, 8),
            3: (2, 3),
            4: (5, 7),
            5: (2, 4),
            6: (2, -3),
        },
    },
    {
        "instance_number": 4,
        "Nv": 4,
        "C": 3,
        "depot": (0, 0),
        "customers": {
            1: (-2, 2),
            2: (-5, 8),
            3: (6, 3),
            4: (4, 4),
            5: (3, 2),
            6: (0, 2),
            7: (-2, 3),
            8: (-4, 3),
            9: (2, 3),
            10: (2, 7),
            11: (-2, 5),
            12: (-1, 4),
        },
    },
    {
        "instance_number": 5,
        "Nv": 5,
        "C": 4,
        "depot": (0, 0),
        "customers": {
            1: (-2, 2),
            2: (-5, 8),
            3: (2, 3),
            4: (5, 7),
            5: (2, 4),
            6: (2, -3),
            7: (-4, 1),
            8: (0, 6),
            9: (3, -2),
            10: (-1, 5),
            11: (6, 1),
            12: (-3, 4),
            13: (4, 3),
            14: (-6, 2),
            15: (1, 7),
            16: (5, -1),
            17: (-2, -4),
            18: (3, 6),
            19: (-5, 5),
            20: (0, -2),
        },
    },
    {
        "instance_number": 6,
        "Nv": 5,
        "C": 5,
        "depot": (0, 0),
        "customers": {
            1: (-3, 4),
            2: (-6, 7),
            3: (2, 3),
            4: (5, 8),
            5: (3, 5),
            6: (1, -4),
            7: (-5, 1),
            8: (0, 7),
            9: (4, -2),
            10: (-1, 6),
            11: (6, 2),
            12: (-4, 3),
            13: (7, 5),
            14: (-7, 2),
            15: (1, 9),
            16: (5, -1),
            17: (-2, -5),
            18: (3, 7),
            19: (-6, 5),
            20: (0, -3),
            21: (8, 1),
            22: (-2, 8),
            23: (4, 0),
            24: (-5, -2),
            25: (2, 6),
        },
    },
]

def run_one_trial(instance, neighborhood_qubits, seed):
    customers = instance["customers"]
    q = min(neighborhood_qubits, len(customers))

    init_problem(
        _customers=customers,
        _Nv=instance["Nv"],
        _C=instance["C"],   # keep capacity fixed
        _depot=instance["depot"],
        neighborhood_qubits=q,
        _shots=SHOTS,
        _gamma=GAMMA,
        _layers=LAYERS,
        _seed=seed,
    )

    routes, total_cost = solve(
        do_classical_cleanup=DO_CLASSICAL_CLEANUP,
        verbose=VERBOSE_SOLVE,
    )
    return routes, float(total_cost), q

summary_rows = []
detailed_rows = []

for instance in INSTANCES:
    inst_no = instance["instance_number"]
    Nv = instance["Nv"]
    C = instance["C"]
    n = len(instance["customers"])

    print(f"\n=== Instance {inst_no} | Nv={Nv}, C={C}, n={n} ===")

    qubit_cost_map = {}
    plotted_q = []
    plotted_cost = []

    for trial_idx, q_trial in enumerate(range(1, MAX_QUBITS_TO_TEST + 1), start=1):
        routes, total_cost, q_used = run_one_trial(instance, q_trial, FIXED_SEED)

        detailed_rows.append(
            {
                "instance_number": inst_no,
                "trial": trial_idx,
                "trial_qubits": q_trial,
                "qubits_used": q_used,
                "seed": FIXED_SEED,
                "cost": total_cost,
                "routes": repr(routes),
                "Nv": Nv,
                "C": C,
                "num_customers": n,
            }
        )

        qubit_cost_map[f"trial_{q_trial}"] = total_cost
        plotted_q.append(q_trial)
        plotted_cost.append(total_cost)

        print(f"  Trial {trial_idx} | qubits={q_trial} | cost={total_cost:.4f}")

    summary_row = {
        "instance_number": inst_no,
        "Nv": Nv,
        "C": C,
        "num_customers": n,
    }
    for q in range(1, MAX_QUBITS_TO_TEST + 1):
        summary_row[f"trial_{q}"] = qubit_cost_map.get(f"trial_{q}", "")
    summary_rows.append(summary_row)

    plt.figure(figsize=(8, 5))
    plt.plot(plotted_q, plotted_cost, marker="o")
    for x, y in zip(plotted_q, plotted_cost):
        plt.annotate(f"{y:.2f}", (x, y), textcoords="offset points", xytext=(0, 6), ha="center")

    plt.title(f"Instance {inst_no}: Cost vs Qubits")
    plt.xlabel("Neighborhood qubits")
    plt.ylabel("Total cost")
    plt.xticks(range(1, MAX_QUBITS_TO_TEST + 1))
    plt.grid(True)

    plot_path = PLOTS_DIR / f"instance_{inst_no}_cost_vs_qubits.png"
    plt.tight_layout()
    plt.savefig(plot_path, dpi=200, bbox_inches="tight")
    plt.close()

    print(f"  Saved plot: {plot_path.resolve()}")

summary_fieldnames = [
    "instance_number",
    *[f"trial_{q}" for q in range(1, MAX_QUBITS_TO_TEST + 1)],
    "Nv",
    "C",
    "num_customers",
]

with open(SUMMARY_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=summary_fieldnames)
    writer.writeheader()
    writer.writerows(summary_rows)

with open(DETAILED_CSV, "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "instance_number",
            "trial",
            "trial_qubits",
            "qubits_used",
            "seed",
            "cost",
            "routes",
            "Nv",
            "C",
            "num_customers",
        ],
    )
    writer.writeheader()
    writer.writerows(detailed_rows)

print("\nSaved:")
print(f"  - {SUMMARY_CSV.resolve()}")
print(f"  - {DETAILED_CSV.resolve()}")
print(f"  - plots folder: {PLOTS_DIR.resolve()}")

print("\nSummary table:")
for row in summary_rows:
    vals = [f"{row[f'trial_{q}']:.4f}" if row[f"trial_{q}"] != "" else "" for q in range(1, MAX_QUBITS_TO_TEST + 1)]
    print(f"Instance {row['instance_number']}: {vals}")


=== Instance 1 | Nv=2, C=5, n=3 ===
  Trial 1 | qubits=1 | cost=31.7359
  Trial 2 | qubits=2 | cost=26.1817
  Trial 3 | qubits=3 | cost=21.7445
  Trial 4 | qubits=4 | cost=21.7445
  Trial 5 | qubits=5 | cost=21.7445
  Trial 6 | qubits=6 | cost=21.7445
  Trial 7 | qubits=7 | cost=21.7445
  Saved plot: /Users/shaiverma/Downloads/dqi_qubit_plots/instance_1_cost_vs_qubits.png

=== Instance 2 | Nv=2, C=2, n=3 ===
  Trial 1 | qubits=1 | cost=31.7359
  Trial 2 | qubits=2 | cost=26.1817
  Trial 3 | qubits=3 | cost=26.1817
  Trial 4 | qubits=4 | cost=26.1817
  Trial 5 | qubits=5 | cost=26.1817
  Trial 6 | qubits=6 | cost=26.1817
  Trial 7 | qubits=7 | cost=26.1817
  Saved plot: /Users/shaiverma/Downloads/dqi_qubit_plots/instance_2_cost_vs_qubits.png

=== Instance 3 | Nv=3, C=2, n=6 ===
  Trial 1 | qubits=1 | cost=65.0959
  Trial 2 | qubits=2 | cost=49.4988
  Trial 3 | qubits=3 | cost=49.4988
  Trial 4 | qubits=4 | cost=49.4988
  Trial 5 | qubits=5 | cost=49.4988
  Trial 6 | qubits=6 | cost=49.

In [21]:
from pathlib import Path
import time

SEEDS = [101, 42, 123, 256, 999]

INSTANCE_SHOTS = {
    1: 1024,
    2: 1024,
    3: 2048,
    4: 4096,
    5: 8192,
    6: 8192
}

GAMMA = 0.8
LAYERS = 1
DO_CLASSICAL_CLEANUP = True
VERBOSE_SOLVE = False

INSTANCES = [
    {
        "instance_number": 1,
        "Nv": 2, "C": 5, "depot": (0, 0),
        "customers": {1: (-2, 2), 2: (-5, 8), 3: (2, 3)},
    },
    {
        "instance_number": 2,
        "Nv": 2, "C": 2, "depot": (0, 0),
        "customers": {1: (-2, 2), 2: (-5, 8), 3: (2, 3)},
    },
    {
        "instance_number": 3,
        "Nv": 3, "C": 2, "depot": (0, 0),
        "customers": {
            1: (-2, 2), 2: (-5, 8), 3: (2, 3),
            4: (5, 7), 5: (2, 4), 6: (2, -3),
        },
    },
    {
        "instance_number": 4,
        "Nv": 4, "C": 3, "depot": (0, 0),
        "customers": {
            1: (-2, 2), 2: (-5, 8), 3: (6, 3), 4: (4, 4),
            5: (3, 2), 6: (0, 2), 7: (-2, 3), 8: (-4, 3),
            9: (2, 3), 10: (2, 7), 11: (-2, 5), 12: (-1, 4),
        },
    },
    {
        "instance_number": 5,
        "Nv": 5, "C": 4, "depot": (0, 0),
        "customers": {
            1: (-2, 2), 2: (-5, 8), 3: (2, 3), 4: (5, 7),
            5: (2, 4), 6: (2, -3), 7: (-4, 1), 8: (0, 6),
            9: (3, -2), 10: (-1, 5), 11: (6, 1), 12: (-3, 4),
            13: (4, 3), 14: (-6, 2), 15: (1, 7), 16: (5, -1),
            17: (-2, -4), 18: (3, 6), 19: (-5, 5), 20: (0, -2),
        },
    },
    {
        "instance_number": 6,
        "Nv": 5, "C": 5, "depot": (0, 0),
        "customers": {
            1: (-3, 4), 2: (-6, 7), 3: (2, 3), 4: (5, 8),
            5: (3, 5), 6: (1, -4), 7: (-5, 1), 8: (0, 7),
            9: (4, -2), 10: (-1, 6), 11: (6, 2), 12: (-4, 3),
            13: (7, 5), 14: (-7, 2), 15: (1, 9), 16: (5, -1),
            17: (-2, -5), 18: (3, 7), 19: (-6, 5), 20: (0, -3),
            21: (8, 1), 22: (-2, 8), 23: (4, 0), 24: (-5, -2),
            25: (2, 6),
        },
    },
]

def pretty_print_routes(routes):
    if not routes:
        print("  No routes returned.")
        return

    for i, route in enumerate(routes, start=1):
        route_str = " -> ".join(str(node) for node in route)
        print(f"  Vehicle {i}: {route_str}")

print("Starting DQI dynamic qubit benchmark...\n")

for instance in INSTANCES:
    inst_no = instance["instance_number"]
    customers = instance["customers"]
    shots = INSTANCE_SHOTS[inst_no]
    max_q_possible = min(7, len(customers))

    global_best_cost = float("inf")
    global_best_routes = None
    global_best_q = None
    global_best_seed = None

    print("=" * 70)
    print(f"Instance {inst_no} | Nv={instance['Nv']} | C={instance['C']} | customers={len(customers)}")
    print(f"Searching q = 1 to {max_q_possible} with {shots} shots")
    print("-" * 70)

    for q in range(1, max_q_possible + 1):
        q_best_cost = float("inf")
        q_best_routes = None
        q_best_seed = None

        for seed in SEEDS:
            init_problem(
                _customers=customers,
                _Nv=instance["Nv"],
                _C=instance["C"],
                _depot=instance["depot"],
                neighborhood_qubits=q,
                _shots=shots,
                _gamma=GAMMA,
                _layers=LAYERS,
                _seed=seed,
            )

            routes, cost = solve(
                do_classical_cleanup=DO_CLASSICAL_CLEANUP,
                verbose=VERBOSE_SOLVE,
            )

            cost = float(cost)

            if cost < q_best_cost:
                q_best_cost = cost
                q_best_routes = routes
                q_best_seed = seed

        print(f"q = {q} | best cost = {q_best_cost:.4f} | seed = {q_best_seed}")

        if q_best_cost < global_best_cost:
            global_best_cost = q_best_cost
            global_best_routes = q_best_routes
            global_best_q = q
            global_best_seed = q_best_seed

    print("\nBest DQI result for this instance")
    print(f"  Optimal value: {global_best_cost:.4f}")
    print(f"  Selected q:    {global_best_q}")
    print(f"  Seed:          {global_best_seed}")
    print(f"  Optimal path(s):")
    pretty_print_routes(global_best_routes)
    print()

print("=" * 70)
print("Benchmark complete.")

Starting DQI dynamic qubit benchmark...

Instance 1 | Nv=2 | C=5 | customers=3
Searching q = 1 to 3 with 1024 shots
----------------------------------------------------------------------
q = 1 | best cost = 31.7359 | seed = 101
q = 2 | best cost = 26.1817 | seed = 101
q = 3 | best cost = 21.7445 | seed = 101

Best DQI result for this instance
  Optimal value: 21.7445
  Selected q:    3
  Seed:          101
  Optimal path(s):
  Vehicle 1: 3 -> 2 -> 1

Instance 2 | Nv=2 | C=2 | customers=3
Searching q = 1 to 3 with 1024 shots
----------------------------------------------------------------------
q = 1 | best cost = 31.7359 | seed = 101
q = 2 | best cost = 26.1817 | seed = 101
q = 3 | best cost = 26.1817 | seed = 101

Best DQI result for this instance
  Optimal value: 26.1817
  Selected q:    2
  Seed:          101
  Optimal path(s):
  Vehicle 1: 1 -> 2
  Vehicle 2: 3

Instance 3 | Nv=3 | C=2 | customers=6
Searching q = 1 to 6 with 2048 shots
----------------------------------------------